# CUDA - 粒子模擬

---

# [Part 2.] GPU -> 輸出視覺化GIF

視覺顯示大氣粒子在城市中的擴散情形

**lab_draw.cu** -> **output/frame_0000-0200.csv** -> **particle_animation.gif**

In [1]:
%%writefile lab_draw.cu

#include <iostream>
#include <vector>
#include <fstream>
#include <sstream>
#include <iomanip>
#include <cmath>
#include <cuda_runtime.h>

using namespace std;

struct Particle {
    float x, y;
};

__host__ __device__
float rand(int i, int step, int seed) {
    int n = i * 1973 + step * 9277 + seed * 26699;
    n = (n << 13) ^ n;
    return 1.0f - ((n * (n * n * 15731 + 789221) + 1376312589) & 0x7fffffff) / 1073741824.0f;
}

__global__ void gpu(Particle* p, int N, int step, float dt, float wind_x, float wind_y, float diff, float settle) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float rx = rand(i, step, 1) * diff;
        float ry = rand(i, step, 2) * diff;

        p[i].x += wind_x * dt + rx;
        p[i].y += wind_y * dt + ry - settle * dt;

        if (p[i].x < 0) p[i].x = 0;
        if (p[i].x > 100) p[i].x = 100;
        if (p[i].y < 0) p[i].y = 0;
        if (p[i].y > 100) p[i].y = 100;
    }
}

void save_particles_to_csv(const vector<Particle>& particles, int frame_id) {
    ostringstream filename; //單純拼字串
    filename << "output/frame_" << setw(4) << setfill('0') << frame_id << ".csv";

    ofstream file(filename.str()); //把剛剛組好的檔名字串，拿去開檔案
    file << "x,y\n"; //寫進檔案
    for (const auto& p : particles) {
        file << p.x << "," << p.y << "\n";
    }
    file.close();
}

int main() {
    const int N = 10000;       //粒子數
    const int STEPS = 1000;      //總步數
    const int SAVE_EVERY = 5;   //每幾步輸出一次
    const float dt = 0.1f;

    // 風場：往右上吹
    const float wind_x = 0.75f;
    const float wind_y = 0.25f;

    // 擴散與沉降
    const float diff = 0.6f;
    const float settle = 0.05f;

    vector<Particle> h_particles(N);

    // 初始污染源：城市左下角偏中
    for (int i = 0; i < N; i++) {
        h_particles[i] = {20.0f, 20.0f};
    }

    Particle* d_particles;
    cudaMalloc(&d_particles, N * sizeof(Particle));
    cudaMemcpy(d_particles, h_particles.data(), N * sizeof(Particle), cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize = (N + blockSize - 1) / blockSize;

    int frame_id = 0;

    // 先存初始狀態
    save_particles_to_csv(h_particles, frame_id++);

    for (int t = 0; t < STEPS; t++) {
        gpu<<<gridSize, blockSize>>>(
            d_particles, N, t, dt, wind_x, wind_y, diff, settle
        );
        cudaDeviceSynchronize();

        if ((t + 1) % SAVE_EVERY == 0) {
            cudaMemcpy(h_particles.data(), d_particles, N * sizeof(Particle), cudaMemcpyDeviceToHost);
            save_particles_to_csv(h_particles, frame_id++);
            cout << "Saved frame " << frame_id - 1 << endl;
        }
    }

    cudaFree(d_particles);
    cout << "Simulation finished." << endl;
    return 0;
}

Writing lab_draw.cu


In [2]:
!if not exist output mkdir output
!nvcc -arch=sm_89 lab_draw.cu -o lab_draw
!lab_draw.exe

lab_draw.cu
tmpxft_00006aa8_00000000-10_lab_draw.cudafe1.cpp
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(1331): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(3125): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(1331): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(3125): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(1331): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_ty

C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(1331): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(3125): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(1331): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(3125): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(1331): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����ܪ��r���C�ХH Unicode �榡�x�s�ɮץH�����ƿ�
C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\include\driver_types.h(3125): warning C4819: �ɮקt���L�k�b�ثe�r�X�� (950) �����

Saved frame 1
Saved frame 2
Saved frame 3
Saved frame 4
Saved frame 5
Saved frame 6
Saved frame 7
Saved frame 8
Saved frame 9
Saved frame 10
Saved frame 11
Saved frame 12
Saved frame 13
Saved frame 14
Saved frame 15
Saved frame 16
Saved frame 17
Saved frame 18
Saved frame 19
Saved frame 20
Saved frame 21
Saved frame 22
Saved frame 23
Saved frame 24
Saved frame 25
Saved frame 26
Saved frame 27
Saved frame 28
Saved frame 29
Saved frame 30
Saved frame 31
Saved frame 32
Saved frame 33
Saved frame 34
Saved frame 35
Saved frame 36
Saved frame 37
Saved frame 38
Saved frame 39
Saved frame 40
Saved frame 41
Saved frame 42
Saved frame 43
Saved frame 44
Saved frame 45
Saved frame 46
Saved frame 47
Saved frame 48
Saved frame 49
Saved frame 50
Saved frame 51
Saved frame 52
Saved frame 53
Saved frame 54
Saved frame 55
Saved frame 56
Saved frame 57
Saved frame 58
Saved frame 59
Saved frame 60
Saved frame 61
Saved frame 62
Saved frame 63
Saved frame 64
Saved frame 65
Saved frame 66
Saved frame 67
Save